# Willett 2021 · handwriting T5

## Prepare the data

Run the commands below from the repository root with `bci-repro` active.

### 1. Download

Download `handwritingBCIData.tar.gz` from [Dryad](https://doi.org/10.5061/dryad.wh70rxwmv). Create `studies/20210512_willett/data/raw/dryad_wh70rxwmv/` and save the download as `studies/20210512_willett/data/raw/dryad_wh70rxwmv/dryad_wh70rxwmv__handwritingBCIData.tar.gz`.

### 2. Extract

Create `studies/20210512_willett/data/extracted/dryad_wh70rxwmv/` before running the extraction command.

```text
tar -xzf "studies/20210512_willett/data/raw/dryad_wh70rxwmv/dryad_wh70rxwmv__handwritingBCIData.tar.gz" -C "studies/20210512_willett/data/extracted/dryad_wh70rxwmv"
```

After extraction, check for `studies/20210512_willett/data/extracted/dryad_wh70rxwmv/handwritingBCIData/Datasets/`. The archive also contains `handwritingBCIData/RNNTrainingSteps/` and `handwritingBCIData/BigramLM/`.

### 3. Prepare the model input

Generate memory-mapped NumPy training batches from the `.mat` recordings. This example prepares one session using the released HMM labels.

```text
python shared/tools/setup_upstream.py --study 20210512_willett
python studies/20210512_willett/code/prepare_data.py --phase both --dataDir t5.2019.05.08 --cvPart HeldOutTrials --nParallel 2
```

**Output** `studies/20210512_willett/data/derived/dryad_wh70rxwmv/npy/synthetic_sentences/HeldOutTrials/t5.2019.05.08_syntheticSentences/`. The directory contains `bat_*_inputs.npy`, `bat_*_labels.npy`, and `bat_*_errW.npy`.

**Snippet output** `studies/20210512_willett/data/derived/dryad_wh70rxwmv/pkl/synthetic_snippets/HeldOutTrials/t5.2019.05.08_snippets.pkl`.

### 4. Export the common session file

Write one recording day to the session format shared by every study in this repository. The file stores each sentence with the released 10 ms neural activity and the HMM letter timing of both released partitions, and records the publisher files and the position the sentence came from. This example exports May 8, 2019, the day step 3 prepares.

```text
python studies/20210512_willett/code/export_sessions.py --session t5.2019.05.08
```

**Output** `studies/20210512_willett/data/processed/t5/dryad_wh70rxwmv_2019-05-08.h5`. Add `--list` to see every session and whether it is exported, or use `--all` to export them all.

Check the file structure and trace every trial back to its publisher file with one command.

```text
python shared/tools/validate_session.py studies/20210512_willett/data/processed/t5/dryad_wh70rxwmv_2019-05-08.h5 --source trials
```

**Output** A `PASS` line for the file, ending with `source trace: 102/102 trials re-read and identical`.

In [1]:
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "shared"))
from session_format import read_session

session_file = root / "studies/20210512_willett/data/processed/t5/dryad_wh70rxwmv_2019-05-08.h5"
if session_file.is_file():
    with read_session(session_file) as session:
        print(f"{session.attrs['session']}: {len(session)} trials")
        trial = next(t for t in session.trials if t.attrs["labeled"] and not t.attrs["excluded"])
        print(trial.trial_id, "·", trial.target("sentence"))
        array = trial.neural("neural_activity")
        print(f"  neural/neural_activity: {array.shape} {array.dtype}")
        chars = session.label_sets["chars"]
        starts = trial.timing("char")[:]
        print("  first characters and their start frames:",
              [(chars[i], int(a)) for i, a in zip(trial.target("char_class")[:5], starts[:5, 0])])
else:
    print("Session file not found. Run the export command above to create it.")

t5.2019.05.08: 102 trials
t5.2019.05.08/data/0000 · thanks>so>much,>honey~
  neural/neural_activity: (1961, 192) uint8
  first characters and their start frames: [('t', 39), ('h', 159), ('a', 209), ('n', 304), ('k', 379)]


## Read the archived results

Use the released HeldOutTrials indices. Figure 3b pairs greedy CER extracted from the PyTorch inference MAT files with bigram word errors on those same trials. The per-trial character errors in the decode CSV describe its bigram output.

This walkthrough uses the checked-in archive and takes no data download.

In [2]:
import csv
import json
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "figures/code"))
from figure_paths import PUBLISHED, load_benchmark

benchmark = load_benchmark()

def read_rows(path):
    """Read the archived CSV rows in their original order."""
    with (root / path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

## From a trial or day to the reported result

In [3]:
from build_session_cer_vs_wer import build_willett21
source = benchmark["figure3_cer_sources"]["Willett 2021 (T5)"]
row = read_rows(source["archive"])[0]
print(row["trueText"], "→", row["decText"])
print(f"Trial bigram word errors: {row['wordErrs']}/{row['wordCounts']}")
result = build_willett21()
print("First recording day:", result["sessions"][0])
print(f"{len(result['sessions'])} sessions, {sum(d['n'] for d in result['sessions'])} trials; pooled {result['pooled']}")

you want me to sing? → 's you want me to sing?
Trial bigram word errors: 1/5
First recording day: {'session': 't5.2019.05.08', 'n': 10, 'cer': 4.47, 'wer': 5.88, 'char_counts': 693, 'word_counts': 136, 'char_errors': 31, 'word_errors': 8}
10 sessions, 63 trials; pooled {'cer': 4.71, 'wer': 10.0}


## Continue the full reproduction

For a separate data disk, set `BCI_DATA_ROOT` to the directory that contains the study folders and use the same layout under that root.

See the [evaluation and input guide](docs/20261124_evaluation_protocol/README.md) and [code guide](code/README.md).

**Public commands** [`prepare_data.py`](code/prepare_data.py), [`export_sessions.py`](code/export_sessions.py), [`train.py`](code/train.py), [`infer.py`](code/infer.py), [`decode.py`](code/decode.py), [`evaluate.py`](code/evaluate.py). Run a command with `--help` to see its arguments.